In [0]:
from pyspark.sql.functions import col, to_timestamp, from_unixtime

In [0]:
dbutils.fs.ls("dbfs:/databricks-datasets/retail-org/customers/")

In [0]:
df_sampled = spark.\
    read.format("csv")\
    .option("header", True)\
    .option("inferSchema", True)\
    .option("samplingRatio", "0.1")\
    .load("dbfs:/databricks-datasets/retail-org/customers/customers.csv")


In [0]:
df_customer = spark.\
    read.format("csv")\
    .option("header", "true")\
    .schema(df_sampled.schema)\
    .load("dbfs:/databricks-datasets/retail-org/customers/customers.csv")

display(df_customer.limit(100))

In [0]:
display(df_customer.summary())

In [0]:
df = df_customer.withColumn(
    "valid_from_ts",
    to_timestamp(from_unixtime(col("valid_from").cast("bigint")))
).withColumn(
    "valid_to_ts",
    to_timestamp(
        from_unixtime(
            col("valid_to").cast("double").cast("bigint")
        )
    )
)

In [0]:
df = (
    df
    .drop("valid_from", "valid_to")
    .withColumnRenamed("valid_from_ts", "valid_from")
    .withColumnRenamed("valid_to_ts", "valid_to")
)

In [0]:
display(df.limit(100))

In [0]:
jdbc_url = "jdbc:sqlserver://ozan-sql-dev-001.database.windows.net:1433;database=free-sql-db-7764205;encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;"
connection_properties = {
    "user": "ozoezer",
    "password": dbutils.secrets.get("sql-ozoezer", "sql-password"),
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

In [0]:
df.write \
    .mode("overwrite") \
    .jdbc(url=jdbc_url, table="customers", properties=connection_properties)

In [0]:
storage_account = "ozandatalake001"
container = "retail"
storage_key = dbutils.secrets.get("sql-ozoezer", "blob-password")

In [0]:
df.write.mode("overwrite").json(
  "abfss://retail@ozandatalake001.dfs.core.windows.net/customers_json/"
)